# MATMEK-4270 — Mandatory assignment 1: Report

This report answers Sections 1.2.3–1.2.6 of the assignment. The solvers are implemented in `poisson2d.py` and `Wave2D.py` in the root of the repository.

## 1.2.3 Exact solution

Consider

$$
u(t,x,y) = e^{\imath (k_x x + k_y y - \omega t)} .
$$

Differentiating,

$$
\frac{\partial^2 u}{\partial t^2} = (-\imath\omega)^2 u = -\omega^2 u,
\qquad
\frac{\partial^2 u}{\partial x^2} = (\imath k_x)^2 u = -k_x^2 u,
\qquad
\frac{\partial^2 u}{\partial y^2} = (\imath k_y)^2 u = -k_y^2 u .
$$

Inserting into $u_{tt} = c^2 \nabla^2 u$:

$$
-\omega^2 u = -c^2 (k_x^2 + k_y^2)\, u .
$$

Since $u \neq 0$ everywhere, the wave equation is satisfied if and only if

$$
\boxed{\omega = c\sqrt{k_x^2 + k_y^2}}
$$

(the dispersion relation, used in the property `Wave2D.w`). Because the equation is linear and $\omega$ depends only on $k_x^2 + k_y^2$, any sum of such waves with $(\pm k_x, \pm k_y)$, and its real and imaginary parts, is also a solution. For example,

$$
\cos(k_x x)\cos(k_y y)\cos(\omega t) = \tfrac14 \sum_{s_x=\pm1}\sum_{s_y=\pm1} \mathrm{Re}\, e^{\imath (s_x k_x x + s_y k_y y - \omega t)},
$$

which is the Neumann standing wave (1.5). The Dirichlet standing wave (1.4) is obtained in the same way.

## 1.2.4 Dispersion coefficient

With $k_x = k_y = k$, insert the discrete wave

$$
u^n_{ij} = e^{\imath (kh(i+j) - \tilde{\omega} n\Delta t)}
$$

into the scheme (1.3).

**Time difference.** Since $u^{n\pm1}_{ij} = e^{\mp \imath\tilde\omega\Delta t} u^n_{ij}$,

$$
u^{n+1}_{ij} - 2u^n_{ij} + u^{n-1}_{ij}
= \left(e^{-\imath\tilde\omega\Delta t} - 2 + e^{\imath\tilde\omega\Delta t}\right) u^n_{ij}
= \left(2\cos(\tilde\omega\Delta t) - 2\right) u^n_{ij}
= -4\sin^2\!\left(\frac{\tilde\omega\Delta t}{2}\right) u^n_{ij}.
$$

**Space differences.** Since $u^n_{i\pm1,j} = e^{\pm \imath kh} u^n_{ij}$,

$$
u^n_{i+1,j} - 2u^n_{ij} + u^n_{i-1,j} = \left(2\cos(kh) - 2\right) u^n_{ij} = -4\sin^2\!\left(\frac{kh}{2}\right) u^n_{ij},
$$

and identically in the $y$-direction.

**Inserting into (1.3)** and dividing by $u^n_{ij} \neq 0$:

$$
-\frac{4}{\Delta t^2}\sin^2\!\left(\frac{\tilde\omega\Delta t}{2}\right)
= -c^2\,\frac{8}{h^2}\sin^2\!\left(\frac{kh}{2}\right)
\quad\Longrightarrow\quad
\sin^2\!\left(\frac{\tilde\omega\Delta t}{2}\right) = 2C^2 \sin^2\!\left(\frac{kh}{2}\right),
$$

with $C = c\Delta t/h$. Taking the branch that tends to the exact $\omega$ as $h \to 0$ (and assuming a resolved wave, $kh \le \pi$):

$$
\sin\!\left(\frac{\tilde\omega\Delta t}{2}\right) = \sqrt{2}\,C \sin\!\left(\frac{kh}{2}\right).
$$

**For $C = 1/\sqrt{2}$** the right-hand side is $\sin(kh/2)$, so

$$
\frac{\tilde\omega\Delta t}{2} = \frac{kh}{2}
\quad\Longrightarrow\quad
\tilde\omega = \frac{kh}{\Delta t} = \frac{kh\,c}{C h} = \sqrt{2}\,c k .
$$

The exact dispersion coefficient with $k_x = k_y = k$ is $\omega = c\sqrt{k^2 + k^2} = \sqrt{2}\,c k$. Hence

$$
\boxed{\tilde\omega = \omega \quad \text{for } C = 1/\sqrt{2},\ m_x = m_y.}
$$

## 1.2.5 Test for exact solution

`test_exact_wave2d` in `Wave2D.py` runs both the Dirichlet and the Neumann solver with $C = 1/\sqrt{2}$ and $m_x = m_y \in \{1, 2, 3\}$, and asserts that the $\ell^2$-error is below $10^{-12}$ at **every** time step.

The error is at round-off level, and not only $\tilde\omega = \omega$, for two reasons:

1. The standing waves are exact eigenvectors of the discrete Laplacian. For Dirichlet this holds since $\sin(k x_i)$ vanishes at the boundary. For Neumann, the ghost-point condition $u_{-1} = u_1$ (boundary row $(-2, 2)$ in `D2`) is exactly satisfied by $\cos(k x_i)$, since $\cos(-kh) = \cos(kh)$.
2. The first step $U^1 = U^0 + \tfrac12 (c\Delta t)^2 \nabla_h^2 U^0$ multiplies the mode by $1 - 2C^2\cdot 2\sin^2(kh/2) = 1 - 2\sin^2(\tilde\omega\Delta t/2) = \cos(\tilde\omega\Delta t)$, i.e. it is the exact discrete solution at $n=1$.

Verification (the last line shows that $C = 0.5$ is not exact):

In [1]:
import sys
sys.path.append("..")
import numpy as np
from Wave2D import Wave2D, Wave2D_Neumann

cfl = 1 / np.sqrt(2)
for sol in (Wave2D(), Wave2D_Neumann()):
    for m in (1, 2, 3):
        _, err = sol(N=20, Nt=50, cfl=cfl, mx=m, my=m, store_data=-1)
        print(f"{type(sol).__name__:15s} mx=my={m}:  max l2-error = {err.max():.2e}")

# For comparison: C = 0.5 (same mx = my = 2) is not exact
_, err = Wave2D()(N=20, Nt=50, cfl=0.5, mx=2, my=2, store_data=-1)
print(f"Wave2D          C=0.5, mx=my=2: max l2-error = {err.max():.2e}")

Wave2D          mx=my=1:  max l2-error = 5.07e-16
Wave2D          mx=my=2:  max l2-error = 7.87e-16
Wave2D          mx=my=3:  max l2-error = 2.95e-15
Wave2D_Neumann  mx=my=1:  max l2-error = 8.43e-16
Wave2D_Neumann  mx=my=2:  max l2-error = 1.83e-15
Wave2D_Neumann  mx=my=3:  max l2-error = 3.35e-15
Wave2D          C=0.5, mx=my=2: max l2-error = 1.14e-02


## 1.2.6 Movie

Neumann problem with $m_x = m_y = 2$, $C = 1/\sqrt{2}$, $N = 40$, two periods ($T = 2\pi/\omega = 1/\sqrt{2}$ for $c = 1$). Created with `make_movie()` in `Wave2D.py`.

![Neumann standing wave](movie.gif)